In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OllamaEmbeddings(model="nomic-embed-text")

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
# STEP 1 - Loading documents of external knowledge base
loader = PyPDFLoader("C:\\Anand\\old_laptop_backup\\D drive data\\Anand\\material\\Training\\Gen_AI\\L2_Applied_Gen_AI\\external_knowledgebase_for_rag\\Python_Programming.pdf")
pages = loader.load()
print(f'No of page created: {len(pages)}')

No of page created: 140


In [3]:
# STEP 2 - Break every document into small chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

docs = spliter.split_documents(pages)
print(f'Document size: {len(docs)}')


Document size: 528


In [ ]:
# STEP 3 - Create embeddings/vectors
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(docs, embedding=embedding_model)
print(f'Vector store: {vector_store}')

In [5]:
# STEP 3 - Create embeddings & vector stores using Chroma

from langchain_community.vectorstores import Chroma

# Define the directory to store ChromaDB data
persist_directory = "c:/content/chroma_db"

# Create a Chroma vector store from the text chunks and embeddings, and persist it to disk
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=persist_directory
)
print(type(vector_store))
print(f"Embeddings successfully stored in ChromaDB and persisted to {persist_directory}.")
print(vector_store)

<class 'langchain_community.vectorstores.chroma.Chroma'>
Embeddings successfully stored in ChromaDB and persisted to c:/content/chroma_db.


In [6]:
# STEP 4 - Integrate embeddings with LLM & build retrieval chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
custom_prompt_template = PromptTemplate(
    template="""Use the following pieces of context to answer the user's question.
    If you don't know the answer of the question then just say you don't know but don't try to make sure an answer
    ------------
    Context: {context}
    Question: {question}
    """,
    input_variables=["context", "question"]    
)
parser = StrOutputParser()
qa_chain = RetrievalQA.from_chain_type(
model,
chain_type="stuff",
retriever=vector_store.as_retriever(),
return_source_documents=True,
chain_type_kwargs={"prompt": custom_prompt_template}
)


In [ ]:
# STEP 5 - User Interface
while True:
    question = input("\nEnter question or type 'quit'")

    if question.lower()=='quit':
        break
    response = qa_chain({"query": question})
    answer = response["result"]
    source_documents = response["source_documents"]

    print(f'Bot: {answer}')
    print(f'Shorlisted documents: {source_documents}')

C:\Users\ak60492\AppData\Local\Temp\ipykernel_29156\299106331.py:6: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_chain({"query": question})


Bot: Here’s a useful MATLAB‑related URL from the provided context:

- https://www.halvorsen.blog/documents/programming/matlab/

If you’re looking for the official MATLAB product page, you can also visit:

- https://www.mathworks.com/products/matlab/
Shorlisted documents: [Document(metadata={'trapped': '/False', 'creator': 'TeX', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'moddate': '2020-08-12T08:23:16+00:00', 'source': 'C:\\Anand\\old_laptop_backup\\D drive data\\Anand\\material\\Training\\Gen_AI\\L2_Applied_Gen_AI\\external_knowledgebase_for_rag\\Python_Programming.pdf', 'page_label': '18', 'creationdate': '2020-08-12T08:23:16+00:00', 'total_pages': 140, 'page': 17, 'producer': 'pdfTeX-1.40.18'}, page_content='https://www.halvorsen.blog/documents/programming/matlab/\n15'), Document(metadata={'producer': 'pdfTeX-1.40.18', 'creationdate': '2020-08-12T08:23:16+00:00', 'page': 46, 'creator': 'TeX', 'ptex.fullbanner': 'This 